# 14: Check a quantum gradient three ways

**Level:** Advanced  
**Before you start:** Notebook 03; derivatives.  
**Resources:** CPU unless an optional remote step is enabled.

How can you tell whether a gradient is correct before trusting a training run?

Run each cell in order. All core calculations are written in this notebook.

## 1. Define a scalar quantity

For a single RY angle and Z observable, f(theta)=cos(theta), so its derivative is -sin(theta).

In [ ]:
import math
import torch
import flagquantum as fq
import matplotlib.pyplot as plt


def value(theta):
    q = fq.Circuit(1).ry(0, theta)
    return fq.run(q, outputs=fq.expectation(fq.Z(0))).expectation().sum()


theta = torch.tensor(0.73, requires_grad=True)
y = value(theta)
y.backward()
autograd = theta.grad.item()
exact = -math.sin(theta.item())
print("Autograd:", autograd, "Analytic:", exact)
assert abs(autograd - exact) < 1e-5


## 2. Use a parameter-shift identity

For this particular rotation, the derivative is half the difference between values at theta+π/2 and theta-π/2. This identity is not a universal formula for arbitrary parameters.

In [ ]:
t = theta.detach()
shift = ((value(t + math.pi / 2) - value(t - math.pi / 2)) / 2).item()
assert abs(shift - autograd) < 1e-5
print("Parameter shift:", shift)


## 3. Explore finite differences

A small step reduces approximation error, but a step that is too small can lose accuracy through floating-point cancellation.

In [ ]:
steps = [10.0 ** (-k) for k in range(1, 8)]
errors = []
for h in steps:
    derivative = ((value(t + h) - value(t - h)) / (2 * h)).item()
    errors.append(abs(derivative - exact))
plt.loglog(steps, [max(e, 1e-12) for e in errors], "o-")
plt.xlabel("Finite-difference step")
plt.ylabel("Absolute derivative error")
plt.show()


## Make it yours

Detach the parameter before building the circuit and inspect requires_grad on the output. Explain why converting a trainable tensor with .item() too early breaks training. Then try a circuit with two independent parameters.